# 🇷🇺➡️ Task B: Русско-Абхазский переводчик

Fine-tuning **NLLB-200-distilled-600M** на параллельном корпусе (184K пар)

**Подход**: добавляем языковой токен `abk_Cyrl`, дообучаем с LoRA, инференс с beam search.

In [ ]:
!pip install -q transformers datasets peft accelerate sacrebleu sentencepiece protobuf bitsandbytes scikit-learn

## 1. Подготовка рабочего пространства

In [ ]:
import os

# Клонируем репозиторий
if not os.path.exists("./workspace"):
    !git clone https://github.com/weissv/task1 ./workspace

%cd ./workspace/task2

# Проверяем наличие корпуса
corpus_path = "corps/ab-ru-parallel.csv"
if os.path.exists(corpus_path):
    print(f"\u2705 Корпус найден: {corpus_path}")
    !wc -l {corpus_path}
else:
    print("\u274c Корпус не найден! Загрузите ab-ru-parallel.csv в corps/")


## 2. Загрузка и очистка данных

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("corps/ab-ru-parallel.csv")
print(f"Всего строк: {len(df)}")
print(df.head())

# Очистка
df = df.dropna()
df = df.drop_duplicates()
df = df[df["ru"].str.len() > 2]
df = df[df["ab"].str.len() > 2]
df = df[df["ru"].str.len() < 500]
df = df[df["ab"].str.len() < 500]
df = df[df["ru"] != df["ab"]]

print(f"\nПосле очистки: {len(df)} строк")
print(f"Средняя длина ru: {df['ru'].str.len().mean():.0f} символов")
print(f"Средняя длина ab: {df['ab'].str.len().mean():.0f} символов")


## 3. Подготовка Dataset для обучения

In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.05, random_state=42)
print(f"Train: {len(train_df)}, Val: {len(val_df)}")

train_dataset = Dataset.from_pandas(train_df[["ru", "ab"]].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[["ru", "ab"]].reset_index(drop=True))
print(train_dataset)
print(val_dataset)


## 4. Загрузка модели и токенизатора

Добавляем новый языковой токен `abk_Cyrl` для абхазского.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang="rus_Cyrl")

# Добавляем абхазский язык
NEW_LANG = "abk_Cyrl"
if NEW_LANG not in tokenizer.additional_special_tokens:
    new_special_tokens = tokenizer.additional_special_tokens + [NEW_LANG]
    tokenizer.add_special_tokens({"additional_special_tokens": new_special_tokens})
    print(f"\u2705 Добавлен токен: {NEW_LANG}")
else:
    print(f"Токен {NEW_LANG} уже существует")

abk_id = tokenizer.convert_tokens_to_ids(NEW_LANG)
print(f"ID токена abk_Cyrl: {abk_id}")

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))
print(f"\u2705 Модель загружена, embeddings: {len(tokenizer)} токенов")


## 5. Токенизация данных

In [ ]:
MAX_LENGTH = 128

def preprocess_function(examples):
    tokenizer.src_lang = "rus_Cyrl"
    inputs = tokenizer(
        examples["ru"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False
    )
    tokenizer.src_lang = "abk_Cyrl"
    labels = tokenizer(
        examples["ab"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False
    )
    inputs["labels"] = labels["input_ids"]
    tokenizer.src_lang = "rus_Cyrl"
    return inputs

print("Токенизация train...")
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
print("Токенизация val...")
tokenized_val = val_dataset.map(preprocess_function, batched=True, remove_columns=val_dataset.column_names)

print(f"\n\u2705 Train: {len(tokenized_train)}, Val: {len(tokenized_val)}")
print("Пример:")
print(tokenizer.decode(tokenized_train[0]["input_ids"]))
print("\u2192")
print(tokenizer.decode(tokenized_train[0]["labels"]))


## 6. Настройка LoRA

Дообучаем только ~2-5% параметров — быстро, эффективно, без переобучения.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 7. Обучение

⏱️ ~30-60 минут на T4 (Colab Free)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb-abkhaz-checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_steps=200,
    lr_scheduler_type="cosine",
    fp16=True,
    group_by_length=True,
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    predict_with_generate=False,
    logging_steps=50,
    report_to="none",
    dataloader_num_workers=0,
    remove_unused_columns=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("\U0001f680 Начинаем обучение...")
trainer.train()
print("\u2705 Обучение завершено!")


## 8. Сохранение модели

Мержим LoRA адаптеры и сохраняем финальные веса.

In [ ]:
import os

merged_model = model.merge_and_unload()
print("\u2705 LoRA адаптеры смержены")

SAVE_DIR = "./weights"
os.makedirs(SAVE_DIR, exist_ok=True)
merged_model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"\u2705 Модель сохранена в {SAVE_DIR}/")
for f in sorted(os.listdir(SAVE_DIR)):
    size_mb = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1024 / 1024
    print(f"  {f} ({size_mb:.1f} MB)")


## 9. Быстрый тест перевода

In [ ]:
import torch

test_tokenizer = AutoTokenizer.from_pretrained("./weights", src_lang="rus_Cyrl")
test_model = AutoModelForSeq2SeqLM.from_pretrained("./weights").cuda()
test_model.eval()

test_sentences = [
    "Это пример текста для перевода!",
    "Абхазский язык — один из древнейших языков мира",
    "Кириллица стала основой абхазской письменности в 1954 году",
    "В абхазском языке насчитывается свыше 80 звуков",
    "По данным на 2021 год, в Абхазии на абхазском языке говорило около 100 тысяч человек",
]

abk_id = test_tokenizer.convert_tokens_to_ids("abk_Cyrl")
print(f"abk_Cyrl token ID: {abk_id}\n")

for sent in test_sentences:
    inputs = test_tokenizer(sent, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = test_model.generate(
            **inputs,
            forced_bos_token_id=abk_id,
            max_length=256,
            num_beams=5,
            length_penalty=1.0,
            no_repeat_ngram_size=3
        )
    translation = test_tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"RU: {sent}")
    print(f"AB: {translation}")
    print("---")


## 10. Оценка BLEU на валидационном сете

In [ ]:
import sacrebleu
import random

random.seed(42)
val_samples = val_df.sample(n=min(100, len(val_df)), random_state=42)

scores = []
for _, row in val_samples.iterrows():
    inputs = test_tokenizer(row["ru"], return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = test_model.generate(
            **inputs,
            forced_bos_token_id=abk_id,
            max_length=256,
            num_beams=5,
            no_repeat_ngram_size=3
        )
    hyp = test_tokenizer.decode(outputs[0], skip_special_tokens=True)
    bleu = sacrebleu.sentence_bleu(hyp, [row["ab"]]).score
    scores.append(bleu)

print(f"\n\U0001f3af Средний BLEU на {len(scores)} примерах: {np.mean(scores):.2f}")
print(f"Медианный BLEU: {np.median(scores):.2f}")
print(f"Мин: {np.min(scores):.2f}, Макс: {np.max(scores):.2f}")


## 11. Генерация solution.py для контеста

In [ ]:
%%writefile solution.py
import json
import pickle
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

tokenizer = AutoTokenizer.from_pretrained("./weights", src_lang="rus_Cyrl")
model = AutoModelForSeq2SeqLM.from_pretrained("./weights").cuda()
model.eval()

with open("input.pickle", "rb") as f:
    data = pickle.load(f)

tgt_lang_id = tokenizer.convert_tokens_to_ids("abk_Cyrl")

results = []
BATCH_SIZE = 8

for i in range(0, len(data), BATCH_SIZE):
    batch = data[i:i+BATCH_SIZE]
    texts = [item["src"] for item in batch]
    
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=tgt_lang_id,
            max_length=256,
            num_beams=5,
            length_penalty=1.0,
            no_repeat_ngram_size=3
        )
    
    translations = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    for item, trans in zip(batch, translations):
        results.append({"rid": item["rid"], "translation": trans})

with open("output.json", "w") as f:
    json.dump(results, f, ensure_ascii=False)

print(f"Переведено {len(results)} предложений")


## 12. Локальная валидация (evaluate.py)

In [ ]:
%%writefile evaluate.py
import json, os, pickle, subprocess, sys

try:
    import sacrebleu
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "sacrebleu"])
    import sacrebleu

EXAMPLES = [
    {
        "rid": 0,
        "src": "Это пример текста для перевода!",
        "ref": "Ари аиҭагаразы атекст аҿырԥштəы ауп!"
    },
    {
        "rid": 1,
        "src": "Абхазский язык — один из древнейших языков мира",
        "ref": "Аԥсуа бызшәа — адунеи аҿы ижәытәӡатәиу абызшәақәа ируакуп"
    },
    {
        "rid": 2,
        "src": "Кириллица стала основой абхазской письменности в 1954 году",
        "ref": "Акириллица аԥсуа ҩыра шьаҭас иаиуит 1954 шықәсазы"
    },
    {
        "rid": 3,
        "src": "В абхазском языке насчитывается свыше 80 звуков",
        "ref": "Аԥсуа бызшәаҿы 80 бжьы иреиҳауп"
    },
    {
        "rid": 4,
        "src": "По данным на 2021 год, в Абхазии на абхазском языке говорило около 100 тысяч человек",
        "ref": "2021 шықәсазы иҟоу аинформациа ала, Аԥсны аԥсышәала ицәажәон 100 нызқьҩык ауаа раҟара"
    }
]

def main():
    inp = [{"rid": e["rid"], "src": e["src"]} for e in EXAMPLES]
    with open("input.pickle", "wb") as f:
        pickle.dump(inp, f)

    print("=" * 70)
    print("ЗАПУСК SOLUTION.PY")
    print("=" * 70)

    res = subprocess.run([sys.executable, "solution.py"], capture_output=True, text=True)
    if res.returncode != 0:
        print(f"ERROR: {res.stderr}")
        return

    with open("output.json", "r") as f:
        outputs = json.load(f)

    out_map = {o["rid"]: o["translation"] for o in outputs}

    scores = []
    print("\n" + "=" * 70)
    print("ДЕТАЛЬНЫЙ АНАЛИЗ ПЕРЕВОДОВ")
    print("=" * 70)

    for e in EXAMPLES:
        hyp = out_map.get(e["rid"], "")
        ref = e["ref"]
        bleu = sacrebleu.sentence_bleu(hyp, [ref]).score
        scores.append(bleu)

        abk_chars = set("ӷӡқҟԥҭҳҵҷҽҿҩџәҕҧ")
        has_abk = any(c in abk_chars for c in hyp)
        status = "ABK" if has_abk else "NOT ABK!"

        print(f"RID: {e['rid']} | BLEU: {bleu:.2f} | {status}")
        print(f"  [SRC]: {e['src']}")
        print(f"  [REF]: {ref}")
        print(f"  [HYP]: {hyp}")
        print("-" * 70)

    avg_bleu = sum(scores) / len(scores)
    print(f"\nИТОГОВЫЙ СРЕДНИЙ BLEU-SCORE: {avg_bleu:.2f}")

if __name__ == "__main__":
    main()


## 13. Запуск локальной валидации

In [ ]:
!python evaluate.py

## 14. Коммит и пуш

In [ ]:
!git add solution.py evaluate.py Dockerfile
!git status
# !git commit -m "task2: NLLB fine-tuned on Abkhaz parallel corpus"
# !git push origin main
